# 2.8 · 功效与样本量 / Power & Sample Size

> **课程定位**
> 实验设计的"算账课"：**开跑之前**回答"要测出 X 的效应需要多少样本、跑多久"。这是 A/B 测试岗位面试的**硬通货**——"怎么定样本量"几乎必问。
> The budgeting lesson: before launching, how many samples to detect effect X? The hard currency of A/B interviews.

> 💡 **面试相关**
> - "A/B 测试样本量怎么算" ★★★★★
> - "什么是 MDE" ★★★★★
> - "效应量 Cohen's d" ★★★★
> - "为什么实验没测出差异（power 分析角度）" ★★★★

---

## 目录
1. [四象限：错误矩阵与 power ⭐](#1)
2. [power 的几何图景（两座山）](#2)
3. [效应量：Cohen's d 与它的局限](#3)
4. [⭐ 样本量公式：推导 + 心算版](#4)
5. [功效曲线：四个变量的拉锯](#5)
6. [比例版：A/B 测试的真实算账 ⭐](#6)
7. [模拟法 power 分析（万能后备）](#7)
8. [⚠ 事后 power 谬误](#8)
9. [实战：设计一个完整 A/B 实验](#9)
10. [小结](#10)


<a id="1"></a>
## 1. 四象限：错误矩阵与 power ⭐

| | $H_0$ 真（无效应）| $H_1$ 真（有效应）|
|---|---|---|
| **拒绝 $H_0$** | Type I（α，假阳）| ✅ **Power = 1−β** |
| **不拒绝** | ✅ 正确 | Type II（β，假阴）|

**惯例配置**：α = 0.05，power = 0.80（β = 0.20）。

注意不对称：**容忍假阴是假阳的 4 倍**（0.20 vs 0.05）——科学界"宁可错过不可冤枉"的传统。业务场景未必该照搬：错过一个真提升（假阴）的代价可能远高于一次误上线。
The 4:1 asymmetry is a scientific-conservatism convention; in business the cost of missing a real win can dwarf a false launch.


<a id="2"></a>
## 2. power 的几何图景 / The Two-Hills Picture

power = $H_1$ 那座"山"落在拒绝域里的面积。四个变量在拉锯：**效应越大 / n 越大 / α 越松 / 噪声越小 → power 越高**。


In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 两座山: H0 下与 H1 下检验统计量的分布
# Two hills: the test statistic under H0 and H1
def plot_power(ax, effect, n, alpha=0.05):
    se = 1 / np.sqrt(n)                       # σ=1 时均值的 SE
    crit = st.norm.ppf(1 - alpha/2) * se      # 双侧拒绝域边界
    xs = np.linspace(-4*se, effect + 4*se, 400)
    h0, h1 = st.norm.pdf(xs, 0, se), st.norm.pdf(xs, effect, se)
    ax.plot(xs, h0, "C0", label="under $H_0$"); ax.plot(xs, h1, "C1", label="under $H_1$")
    ax.fill_between(xs, h1, where=xs > crit, alpha=0.4, color="C1")
    ax.axvline(crit, color="k", ls="--", lw=1)
    power = st.norm.sf(crit, effect, se) + st.norm.cdf(-crit, effect, se)
    ax.set_title(f"effect={effect}, n={n} → power={power:.0%}", fontsize=10)
    ax.set_yticks([])
    return power

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
plot_power(axes[0], effect=0.3, n=50)
plot_power(axes[1], effect=0.3, n=200)        # 加 n → 山变瘦 → 重叠减少
plot_power(axes[2], effect=0.6, n=50)         # 加效应 → 山分开
axes[0].legend(fontsize=9)
plt.suptitle("Power = orange area beyond the critical line", y=1.05)
plt.tight_layout(); plt.show()


**读图**：橙色阴影 = power。加 $n$ 让两座山**变瘦**（SE 缩小），加效应让两座山**分开**——殊途同归都是减少重叠。
Adding n makes the hills skinnier; bigger effects pull them apart — both shrink the overlap.


<a id="3"></a>
## 3. 效应量：Cohen's d / Effect Size

power 公式需要"标准化的效应"：
$$d = \frac{\mu_1 - \mu_0}{\sigma}$$

| d | 标签（Cohen 1988）| 直觉 |
|---|---|---|
| 0.2 | small | 两组分布重叠 ~85% |
| 0.5 | medium | 肉眼勉强可见 |
| 0.8 | large | 明显分开 |

⚠ **标签的局限**：工业界 d=0.02 的转化率提升可能值几千万美元——**"小"效应不等于"不重要"**，重要性由业务定，Cohen 标签只在没有业务标尺时垫底用。
A d=0.02 conversion lift can be worth millions. Importance comes from the business, not the label.


<a id="4"></a>
## 4. ⭐ 样本量公式：推导 + 心算版 / The Formula

**推导**（双侧 α、目标 power $1-\beta$、两组各 $n$）：

要求两个条件同时成立——
1. 拒绝域边界：$c = z_{1-\alpha/2} \cdot \mathrm{SE}$
2. $H_1$ 下越过边界的概率 = $1-\beta$：$c = \Delta - z_{1-\beta} \cdot \mathrm{SE}$

联立（$\mathrm{SE} = \sigma\sqrt{2/n}$，两组比较）：

$$\boxed{\;n_{\text{per group}} = \frac{2\,(z_{1-\alpha/2} + z_{1-\beta})^2\,\sigma^2}{\Delta^2} = \frac{2\,(z_{1-\alpha/2} + z_{1-\beta})^2}{d^2}\;}$$

**心算版**（α=0.05, power=0.80：$(1.96+0.84)^2 = 7.85 \approx 8$）：

$$n \approx \frac{16}{d^2} \quad \text{（每组）}$$

| d | n/组（心算 16/d²）| 精确 |
|---|---|---|
| 0.5 | 64 | 63 |
| 0.2 | 400 | 393 |
| 0.05 | 6400 | 6280 |

**$n \propto 1/d^2$**：想测一半大的效应 → 4 倍样本。这就是 2.3 √n 法则的实验设计版。


In [ ]:
from statsmodels.stats.power import TTestIndPower

solver = TTestIndPower()
print(f"{'d':>6} {'16/d² 心算':>11} {'statsmodels 精确':>17}")
for d in [0.8, 0.5, 0.2, 0.1, 0.05]:
    n_exact = solver.solve_power(effect_size=d, alpha=0.05, power=0.80)
    print(f"{d:>6} {16/d**2:>11.0f} {n_exact:>17.0f}")


<a id="5"></a>
## 5. 功效曲线 / Power Curves

设计实验时画**功效曲线**比单点计算有用——一眼看到"还差多少"。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

# 左: power vs n (不同效应) / power vs n at several effects
ns = np.arange(10, 1000, 10)
for d in [0.1, 0.2, 0.3, 0.5]:
    axes[0].plot(ns, solver.power(effect_size=d, nobs1=ns, alpha=0.05), label=f"d={d}")
axes[0].axhline(0.8, color="k", ls="--", lw=1)
axes[0].set_xlabel("n per group"); axes[0].set_ylabel("power"); axes[0].legend()
axes[0].set_title("Power vs n — small effects need brutal n")

# 右: 最小可检测效应 MDE vs n ⭐
# MDE 解析式 (两组 z 近似): (z_{1-α/2}+z_{1-β})·√(2/n) — 与第 5 节公式一致
mde = (st.norm.ppf(0.975) + st.norm.ppf(0.80)) * np.sqrt(2 / ns)
axes[1].plot(ns, mde, "C3")
axes[1].set_xlabel("n per group"); axes[1].set_ylabel("MDE (Cohen's d)")
axes[1].set_title("MDE vs n — what THIS budget can detect")
plt.tight_layout(); plt.show()


**MDE（Minimum Detectable Effect）⭐**——把问题反过来问："给定预算 $n$、α、power，**能可靠测出的最小效应是多少**？"

$$\mathrm{MDE} = (z_{1-\alpha/2} + z_{1-\beta}) \cdot \sqrt{\frac{2\sigma^2}{n}}$$

**A/B 平台的标准沟通工具**："你这个流量一周只能测出 ≥2% 的相对提升；想测 0.5% 要 16 周——还做吗？"
The standard conversation: "your traffic detects >=2% lifts in a week; 0.5% takes 16 weeks — still want it?"


<a id="6"></a>
## 6. 比例版：A/B 测试的真实算账 ⭐ / Proportions

转化率场景（最常见），$p_1 \to p_2$：
$$n \approx \frac{(z_{1-\alpha/2} + z_{1-\beta})^2\,\big[p_1(1-p_1) + p_2(1-p_2)\big]}{(p_2 - p_1)^2}$$


In [ ]:
import statsmodels.stats.api as sms

def ab_sample_size(p_base, rel_lift, alpha=0.05, power=0.80):
    p2 = p_base * (1 + rel_lift)
    es = sms.proportion_effectsize(p_base, p2)        # Cohen's h (arcsine 变换)
    return sms.NormalIndPower().solve_power(es, alpha=alpha, power=power)

print(f"基线转化率 5%, α=0.05, power=80%:")
print(f"{'相对提升':>9} {'绝对':>8} {'n/组':>12} {'日流量1万/组需':>13}")
for lift in [0.20, 0.10, 0.05, 0.02]:
    n_req = ab_sample_size(0.05, lift)
    print(f"{lift:>8.0%} {0.05*lift:>8.3%} {n_req:>12,.0f} {n_req/10_000:>11.1f} 天")


**残酷的现实**：测 2% 相对提升（0.1pp）要每组 **150 万+** 用户。这解释了——
- 为什么只有大厂能测微小效应（流量就是统计功效）
- 为什么中小公司应该测**大改动**而不是按钮颜色
- 为什么方差削减技术（CUPED、分层）价值连城：等效于免费扩大流量

The brutal math: a 2% relative lift at 5% baseline needs 1.5M+ per arm. Traffic IS statistical power.


<a id="7"></a>
## 7. 模拟法 power 分析 / Simulation-based Power

公式只覆盖标准检验。**非标场景**（重尾指标、整群随机、序贯逻辑、中位数检验）一律用模拟——**按设定的真效应生成假数据 → 跑你真实要用的检验 → 数拒绝比例**：
For anything non-standard, simulate: generate data with the assumed effect, run your actual test, count rejections.


In [ ]:
def sim_power(gen_a, gen_b, test, n, n_sim=2000, alpha=0.05):
    # gen(size) -> 样本; test(a, b) -> p 值
    return np.mean([test(gen_a(n), gen_b(n)) < alpha for _ in range(n_sim)])

# 场景: 重偏态收入指标 + 中位数比较 (Mann-Whitney), 公式没有现成答案
gen_ctrl = lambda n: rng.lognormal(3.0, 1.0, n)
gen_trt  = lambda n: rng.lognormal(3.08, 1.0, n)            # 中位数 +8%
mw_test  = lambda a, b: st.mannwhitneyu(a, b).pvalue

print("lognormal 指标, 中位数 +8%, Mann-Whitney:")
for n_ in [200, 500, 1000, 2000]:
    print(f"  n={n_:>5}/组 → power = {sim_power(gen_ctrl, gen_trt, mw_test, n_):.0%}")


**模拟法是万能后备**，也是检验"公式假设是否成立"的审计工具（和 2.5 覆盖率审计同一思想）。面试说出"非标场景用模拟估 power"是明显加分项。


<a id="8"></a>
## 8. ⚠ 事后 power 谬误 / The Post-hoc Power Fallacy

实验没显著，老板问："power 够吗？" 有人用**观测到的效应**回算 power——得到"观测 power = 30%"，结论"power 不足，效应可能存在"。

**这是循环论证**：观测 power 是 p 值的确定性函数（p 大 ⇔ 观测 power 低），**不含任何新信息**。
Observed power is a deterministic function of the p-value — zero new information.

**正确动作**：
1. 用**设计时设定的 MDE**（或业务最小有意义效应）评估 power——这在看数据前就该完成
2. 报告效应的**置信区间**："差异的 CI = [−0.2%, +1.1%]" 比"不显著"信息量大得多——它直接显示数据排除了哪些效应大小


<a id="9"></a>
## 9. 实战：设计一个完整 A/B 实验 / Hands-on: Full A/B Design

**任务**：结账页改版，基线转化率 4%，业务认为相对提升 ≥5% 才值得上线。日流量 8 万（均分两组）。**写设计文档的统计部分**：


In [ ]:
# === A/B 实验设计文档: 统计部分 / The stats section of the design doc ===
p0, mde_rel, daily_per_arm = 0.04, 0.05, 40_000
p1 = p0 * (1 + mde_rel)

# 1) 样本量 / Sample size
es = sms.proportion_effectsize(p0, p1)
n_req = sms.NormalIndPower().solve_power(es, alpha=0.05, power=0.80)
days = int(np.ceil(n_req / daily_per_arm))

print(f"== 设计参数 ==")
print(f"基线 p0 = {p0:.1%},  MDE = +{mde_rel:.0%} 相对 (→ p1 = {p1:.2%})")
print(f"α = 0.05 (双侧), power = 0.80")
print(f"\n== 计算结果 ==")
print(f"每组需 n = {n_req:,.0f}")
print(f"按日流量 {daily_per_arm:,}/组 → 至少跑 {days} 天")
print(f"建议跑 {max(days, 14)} 天 (≥2 个完整周期, 避免周内效应)")

# 2) 预演决策规则 / Pre-registered decision rule
print(f"\n== 预注册决策规则 ==")
print(f"主指标: 结账转化率 (唯一); 护栏: 页面延迟 (单侧, 恶化>5%即停)")
print(f"到期一次性分析; 期间不偷看 (sequential 方案见 Part 19)")

# 3) 跑完后的假想分析 / Simulated readout
n_act = int(n_req)
conv_a = rng.binomial(n_act, p0); conv_b = rng.binomial(n_act, p1)
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
z, p = proportions_ztest([conv_b, conv_a], [n_act, n_act])
ci = confint_proportions_2indep(conv_b, n_act, conv_a, n_act)
print(f"\n== (模拟) 实验结果 ==")
print(f"A: {conv_a/n_act:.3%}  B: {conv_b/n_act:.3%}  → p = {p:.4f}")
print(f"差值 95% CI = [{ci[0]:+.3%}, {ci[1]:+.3%}]")
print(f"结论: {'上线 (显著且 CI 排除 0)' if p < 0.05 else '不上线; 报告 CI 而非仅 p'}")


**这一段输出就是大厂实验设计文档的统计骨架**——面试被问"怎么设计 A/B"时照此结构回答：基线 + MDE（业务定）→ 样本量/时长 → 预注册决策规则 → 报告 CI。
This output IS the stats skeleton of a real experiment design doc — answer the interview question in this exact structure.


<a id="10"></a>
## 10. 小结 / Summary

```
power = P(测出 | 真有) = 1 - β     惯例: α=0.05, power=0.80

样本量心算: n ≈ 16/d²  每组        n ∝ 1/Δ² — 效应减半, 样本×4
MDE: 反着问 — 这个预算能测出多大效应  ← A/B 平台的沟通货币

比例版残酷现实: 5% 基线测 2% 相对提升 → 150 万/组
非标场景 → 模拟法 (生成→检验→数拒绝率)
事后 power = p 值换皮, 循环论证 → 报告差值 CI
```

### 💡 面试速查
1. **n ≈ 16/d²** 心算公式（α=0.05, power=0.8）
2. **MDE 定义** + 用它和业务方对话
3. **效应减半 → 样本 4 倍**
4. **"没测出"的正确解读**：报差值 CI，不做事后 power
5. **流量就是统计功效**——方差削减 = 免费流量

### 下一节
**2.9 MLE**——0.9 见过 Bernoulli/Normal 的解析解；这次上数值优化、Fisher 信息、渐近标准误——所有"模型训练"的统计学根基。
